# Tiered Remuneration

Act 1 rewritten from the PDF description. The source notebook is used only for the numerical definitions of `u`, `u'`, `rho_0`, and for the axis geometry.

## Act 1 Target

1. Introduce `u` on the full main plot.
2. Shrink that plotted region into the upper half using a theater.
3. Draw the lower panel and introduce `u'` there with clipping.
4. Remove the upper panel and bring attention back to the main stage by enlarging the lower theater.
5. End in the same state as `example-2.ipynb`, including the first `\rho_0` draw/fill.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyle,
    Draw,
    DrawTheater,
    Erase,
    EraseTheater,
    FillBetween,
    FillBetweenArea,
    MoveTheater,
    Parallel,
    Schedule,
    Theater,
)


In [ ]:
DATA = {}

np.random.seed(626)
N = 10000
epsilon = 1e-2
move = 575
uprimecut = 30
g = np.random.randn(N) * 1 / np.sqrt(N)
G = np.cumsum(g)
GG = np.cumsum(epsilon + np.power(np.abs(G), 1.7)) / np.sqrt(N)
X = np.linspace(0, 1, N)
U_prime = 10 - GG + 1 / (X + 0.02)
U = np.cumsum(U_prime) / N - 2
U /= np.max(U)
U_prime /= uprimecut

DATA["u"] = U
DATA["uprime"] = U_prime

GG = np.cumsum(G) / np.sqrt(N)
Y = 2 - GG
Y = Y[::-1] / 15

DATA["rho0"] = Y
DATA["uprime_plus_rho0"] = U_prime + Y

np.random.seed(60)
g = np.random.randn(N) * 1 / np.sqrt(N)
G = np.cumsum(g)
GG = np.cumsum(epsilon + np.power(np.abs(G), 1.7)) / np.sqrt(N) / np.sqrt(N)
Ymonotone = (0.09 - GG) * 1.7

DATA["rho1"] = Ymonotone
DATA["uprime_plus_rho1"] = U_prime + Ymonotone


In [ ]:
arrow_move = 0.03
scaling = 0.55
startup = 0.52
startdown = -0.1
neww = 0.49

main_axis = Curve(
    "main_axis",
    [0.0, 0.0, 1.05 + scaling * (arrow_move - 0.01)],
    [1.05 + arrow_move - 0.01, 0.0, 0.0],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)
main_xhead = Curve(
    "main_xhead",
    [1.05, 1.05 + scaling * arrow_move, 1.05],
    [arrow_move, 0.0, -arrow_move],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)
main_yhead = Curve(
    "main_yhead",
    [scaling * arrow_move, 0.0, -scaling * arrow_move],
    [1.05, 1.05 + arrow_move, 1.05],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)

up_axis = Curve(
    "up_axis",
    [0.0, 0.0, 1.05 + scaling * (arrow_move - 0.01)],
    [startup + neww + arrow_move + 0.05 - 0.01, startup, startup],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)
up_xhead = Curve(
    "up_xhead",
    [1.05, 1.05 + scaling * arrow_move, 1.05],
    [startup + arrow_move, startup, startup - arrow_move],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)
up_yhead = Curve(
    "up_yhead",
    [scaling * arrow_move, 0.0, -scaling * arrow_move],
    [startup + neww + 0.05, startup + neww + 0.05 + arrow_move, startup + neww + 0.05],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)

down_axis = Curve(
    "down_axis",
    [0.0, 0.0, 1.05 + scaling * (arrow_move - 0.01)],
    [startdown + neww + 0.05 + arrow_move - 0.01, startdown, startdown],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)
down_xhead = Curve(
    "down_xhead",
    [1.05, 1.05 + scaling * arrow_move, 1.05],
    [startdown + arrow_move, startdown, startdown - arrow_move],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)
down_yhead = Curve(
    "down_yhead",
    [scaling * arrow_move, 0.0, -scaling * arrow_move],
    [startdown + neww + 0.05, startdown + neww + 0.05 + arrow_move, startdown + neww + 0.05],
    color="black",
    linewidth=2.0,
    alpha=0.0,
)

u_panel = Theater(
    "u_panel",
    xlim=(0.0, 1.0),
    ylim=(0.0, 1.0),
    local_xlim=(0.0, 1.0),
    local_ylim=(0.0, 1.0),
    facecolor="#ffffff",
    edgecolor="none",
    alpha=0.0,
)
uprime_panel = Theater(
    "uprime_panel",
    xlim=(0.0, 1.0),
    ylim=(startdown, startdown + neww),
    local_xlim=(0.0, 1.0),
    local_ylim=(0.0, 1.0),
    facecolor="#ffffff",
    edgecolor="none",
    alpha=0.0,
)

u_curve = Curve(
    "u",
    X,
    DATA["u"],
    theater_id="u_panel",
    color="red",
    linewidth=2.0,
)
uprime_curve = Curve(
    "uprime",
    X,
    DATA["uprime"],
    theater_id="uprime_panel",
    color="blue",
    linewidth=2.0,
)

rho0_curve = Curve(
    "rho0",
    X,
    DATA["rho0"],
    color="green",
    linewidth=2.0,
)
rho0_fill = FillBetweenArea(
    "rho0_fill",
    X,
    DATA["rho0"],
    0.0,
    color="green",
    alpha=0.2,
    linewidth=0.0,
)


In [ ]:
schedule = Schedule()

schedule.add(
    Parallel([
        DrawTheater(u_panel),
        Draw(main_axis),
        Draw(main_xhead),
        Draw(main_yhead),
    ]),
    duration=0.0,
)
schedule.add(
    Parallel([
        CurveStyle("main_axis", alpha=1.0),
        CurveStyle("main_xhead", alpha=1.0),
        CurveStyle("main_yhead", alpha=1.0),
    ]),
    duration=1.0,
)

schedule.add(Draw(u_curve), duration=1.0)

schedule.add(
    Parallel([
        MoveTheater("u_panel", ylim=(startup, startup + neww)),
        CurveStyle("main_axis", alpha=0.0),
        CurveStyle("main_xhead", alpha=0.0),
        CurveStyle("main_yhead", alpha=0.0),
    ]),
    duration=1.0,
)

schedule.add(
    Parallel([
        DrawTheater(uprime_panel),
        Draw(up_axis),
        Draw(up_xhead),
        Draw(up_yhead),
        Draw(down_axis),
        Draw(down_xhead),
        Draw(down_yhead),
    ]),
    duration=0.0,
)
schedule.add(
    Parallel([
        CurveStyle("up_axis", alpha=1.0),
        CurveStyle("up_xhead", alpha=1.0),
        CurveStyle("up_yhead", alpha=1.0),
        CurveStyle("down_axis", alpha=1.0),
        CurveStyle("down_xhead", alpha=1.0),
        CurveStyle("down_yhead", alpha=1.0),
    ]),
    duration=1.0,
)

schedule.add(Draw(uprime_curve), duration=1.0)

schedule.add(
    Parallel([
        CurveStyle("u", alpha=0.0),
        CurveStyle("up_axis", alpha=0.0),
        CurveStyle("up_xhead", alpha=0.0),
        CurveStyle("up_yhead", alpha=0.0),
    ]),
    duration=1.0,
)
schedule.add(Erase("u"), duration=1.0)
schedule.add(EraseTheater("u_panel"), duration=0.0)

schedule.add(
    Parallel([
        MoveTheater("uprime_panel", ylim=(0.0, 1.2), local_ylim=(0.0, 1.2)),
        CurveStyle("down_axis", alpha=0.0),
        CurveStyle("down_xhead", alpha=0.0),
        CurveStyle("down_yhead", alpha=0.0),
    ]),
    duration=1.0,
)
schedule.add(
    Parallel([
        CurveStyle("main_axis", alpha=1.0),
        CurveStyle("main_xhead", alpha=1.0),
        CurveStyle("main_yhead", alpha=1.0),
    ]),
    duration=1.0,
)

schedule.add(
    Parallel([
        Draw(rho0_curve),
        FillBetween(rho0_fill),
    ]),
    duration=1.0,
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.axis("off")
anim = schedule.build_animation(
    fig=fig,
    ax=ax,
    fps=24,
    xlim=(-0.2, 1.2),
    ylim=(-0.2, 1.2),
)
plt.close(fig)

act_1_final_scene = schedule.final_scene
HTML(anim.to_jshtml())
